# Southeastern Australia biodiversity classification

**Project:** Predict `Occurrence Status` from monthly TerraClimate rasters sampled at labeled locations. The attached notebook and its dataset concern biodiversity classification; the challenge name in the accompanying request differs from the contents of this file. This notebook documents the attached analysis as supplied.

**Models included:** Random Forest, LightGBM, XGBoost, and a soft-voting ensemble of the three.

## Reproducibility and data contract

- Use a Python environment with `numpy`, `pandas`, `matplotlib`, `rasterio`, `scikit-learn`, `lightgbm`, and `xgboost`. Install missing packages in your own environment before running this notebook. No API keys or network credentials are required.
- Set `DATA_DIR` to the folder containing `Training_Data.csv`, `Test.csv`, and `terraclimate_YYYYMMDD.tif` files. `OUTPUT_DIR` holds caches, fitted models, and the submission. Kaggle defaults are below and can be overridden with `BIODIVERSITY_DATA_DIR` and `BIODIVERSITY_OUTPUT_DIR` environment variables.
- CSVs must have `Latitude` and `Longitude` in WGS84 degrees; training also needs a binary `Occurrence Status` encoded as 0/1. Rasters must contain 14 bands in the listed order. Verify the band order against the source dataset documentation before interpreting importance scores.
- The submission preserves every test column except latitude, longitude, and any pre-existing `Target`, then appends `Target`. Check this against the actual competition submission template.
- If rows from the same site or spatial neighborhood appear in both splits, the ordinary stratified holdout can overestimate generalization; use spatial or group splitting when location identifiers permit it. No site identifier is assumed here.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform as transform_coordinates
import joblib

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 38
DATA_DIR = Path(os.environ.get(
    "BIODIVERSITY_DATA_DIR",
    "/kaggle/input/datasets/salvavidachitsa/bio-diversity-southeastern-australia",
))
OUTPUT_DIR = Path(os.environ.get("BIODIVERSITY_OUTPUT_DIR", "/kaggle/working"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = "Occurrence Status"
VARIABLES = ["aet", "def", "pdsi", "pet", "pr", "ro", "soil", "srad", "swe", "tmmn", "tmmx", "vap", "vpd", "vs"]
SELECTED_MODEL = "random_forest"  # alternatives: lightgbm, xgboost, ensemble

## 1. Load labels, locations, and rasters

The original notebook listed 25 rasters, spanning November 2017 through November 2019. The explicit expected filenames below prevent an unnoticed change in time coverage when new files appear. Extraction samples raster pixels by geographic coordinates and preserves input row order.

In [ ]:
train_locations = pd.read_csv(DATA_DIR / "Training_Data.csv")
test_locations = pd.read_csv(DATA_DIR / "Test.csv")
required_train = {"Latitude", "Longitude", TARGET}
required_test = {"Latitude", "Longitude"}
if missing := required_train - set(train_locations.columns):
    raise ValueError(f"Missing training columns: {sorted(missing)}")
if missing := required_test - set(test_locations.columns):
    raise ValueError(f"Missing test columns: {sorted(missing)}")
if train_locations[["Latitude", "Longitude", TARGET]].isna().any().any():
    raise ValueError("Training coordinates and labels must be complete")
if test_locations[["Latitude", "Longitude"]].isna().any().any():
    raise ValueError("Test coordinates must be complete")
if not (train_locations["Latitude"].between(-90, 90).all() and
        test_locations["Latitude"].between(-90, 90).all() and
        train_locations["Longitude"].between(-180, 180).all() and
        test_locations["Longitude"].between(-180, 180).all()):
    raise ValueError("Latitude/longitude must be WGS84 decimal degrees")
labels = train_locations[TARGET].astype(int)
if not train_locations[TARGET].eq(labels).all() or set(labels.unique()) != {0, 1}:
    raise ValueError("Occurrence Status must contain both binary labels 0 and 1")
print("Training rows:", len(train_locations), "Test rows:", len(test_locations))
print("Class counts:\n", labels.value_counts().sort_index())

dates = pd.date_range("2017-11-01", "2019-11-01", freq="MS")
raster_paths = [DATA_DIR / f"terraclimate_{date:%Y%m%d}.tif" for date in dates]
missing_rasters = [str(p) for p in raster_paths if not p.is_file()]
if missing_rasters:
    raise FileNotFoundError(f"Missing {len(missing_rasters)} monthly rasters; first: {missing_rasters[0]}")
print(f"Using {len(raster_paths)} monthly rasters from {dates[0]:%Y-%m} to {dates[-1]:%Y-%m}")

### Raster inspection (optional)

A quick band preview helps confirm the geographic coverage and band order. Masked pixels are hidden; the display does not alter model features.

In [ ]:
with rasterio.open(raster_paths[0]) as src:
    if src.count != len(VARIABLES):
        raise ValueError(f"Expected {len(VARIABLES)} bands; got {src.count}")
    fig, axes = plt.subplots(4, 4, figsize=(13, 12))
    for band_idx, (ax, name) in enumerate(zip(axes.flat, VARIABLES), start=1):
        ax.imshow(src.read(band_idx, masked=True), cmap="viridis")
        ax.set_title(name)
        ax.axis("off")
    for ax in list(axes.flat)[len(VARIABLES):]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()

## 2. Build or load temporal features

`sample(masked=True)` reads pixels at true raster coordinates; nodata and points outside the raster become NaN. Each array has shape `(locations, months, bands)`. Caches live alongside the configured output folder. Keep `REBUILD_FEATURES=True` if you change the input CSV order, raster contents, or extraction code: an existing `.npy` cache does not encode those dependencies.

In [ ]:
def extract_temporal_features(locations: pd.DataFrame, paths: list[Path]) -> np.ndarray:
    """Sample each raster in row order; input coordinates are WGS84 lon/lat."""
    lon = locations["Longitude"].to_numpy(dtype=float)
    lat = locations["Latitude"].to_numpy(dtype=float)
    months = []
    for path in paths:
        with rasterio.open(path) as src:
            if src.count != len(VARIABLES):
                raise ValueError(f"{path.name}: expected {len(VARIABLES)} bands, got {src.count}")
            if src.crs is None:
                raise ValueError(f"{path.name}: raster CRS is missing")
            if src.crs == rasterio.crs.CRS.from_epsg(4326):
                xs, ys = lon, lat
            else:
                xs, ys = transform_coordinates("EPSG:4326", src.crs, lon.tolist(), lat.tolist())
            values = np.ma.array(list(src.sample(zip(xs, ys), masked=True)))
            # Use the valid-data mask and check for nonfinite values below.
            months.append(np.asarray(values.astype(float).filled(np.nan), dtype=float))
    return np.stack(months, axis=1)

REBUILD_FEATURES = True
train_cache = OUTPUT_DIR / "temporal_train.npy"
test_cache = OUTPUT_DIR / "temporal_test.npy"
if REBUILD_FEATURES or not (train_cache.exists() and test_cache.exists()):
    temporal_train = extract_temporal_features(train_locations, raster_paths)
    temporal_test = extract_temporal_features(test_locations, raster_paths)
    np.save(train_cache, temporal_train)
    np.save(test_cache, temporal_test)
else:
    temporal_train, temporal_test = np.load(train_cache), np.load(test_cache)
expected_train_shape = (len(train_locations), len(raster_paths), len(VARIABLES))
expected_test_shape = (len(test_locations), len(raster_paths), len(VARIABLES))
if temporal_train.shape != expected_train_shape or temporal_test.shape != expected_test_shape:
    raise ValueError("Feature cache shape does not match the current CSVs and rasters")
print("Temporal shapes:", temporal_train.shape, temporal_test.shape)

### Aggregate monthly rasters

The original tree models used the per-band mean over time. Retain NaNs for locations with no valid raster values; the imputer is fitted only on the training portion inside each pipeline. Coordinates and identifiers are deliberately excluded from model inputs.

In [ ]:
def temporal_mean(values: np.ndarray) -> pd.DataFrame:
    valid = np.isfinite(values)
    sums = np.where(valid, values, 0).sum(axis=1)
    counts = valid.sum(axis=1)
    mean = np.divide(sums, counts, out=np.full(sums.shape, np.nan), where=counts > 0)
    return pd.DataFrame(mean, columns=VARIABLES)

X = temporal_mean(temporal_train)
X_test = temporal_mean(temporal_test)
y = labels.reset_index(drop=True)
print("Missing training feature values:", int(X.isna().sum().sum()))
print("Missing test feature values:", int(X_test.isna().sum().sum()))

## 3. Holdout evaluation

Use a stratified 80/20 split. The holdout is reserved for evaluation; tuning and feature selection must see only the training partition. Report F1, recall, ROC AUC, and confusion matrix. The original notebook observed that ADASYN hurt tree performance, so oversampling is not part of this default pipeline; if retried, it must be fitted within each training fold.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

def build_models() -> dict[str, Pipeline]:
    """Create fresh, independent pipelines for each classical model."""
    classifiers = {
        "random_forest": RandomForestClassifier(
            n_estimators=400, max_depth=10, min_samples_split=5,
            max_features="log2", random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "lightgbm": LGBMClassifier(
            n_estimators=300, learning_rate=0.05, random_state=RANDOM_STATE,
            verbosity=-1, n_jobs=1,
        ),
        "xgboost": XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", n_estimators=300,
            learning_rate=0.05, max_depth=3, tree_method="hist",
            random_state=RANDOM_STATE, n_jobs=1,
        ),
    }
    classifiers["ensemble"] = VotingClassifier(
        estimators=[(name, clone(model)) for name, model in classifiers.items()],
        voting="soft", n_jobs=1,
    )
    return {
        name: Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                        ("model", model)])
        for name, model in classifiers.items()
    }

models = build_models()

### Train and compare all four models

The ensemble averages class probabilities with equal weights. If probability calibration matters, assess it separately before relying on probability thresholds. All four models are fitted on the same training rows and evaluated on the untouched validation rows.

In [ ]:
fitted_holdout = {}
metrics = []
for name, pipeline in models.items():
    fitted = clone(pipeline).fit(X_train, y_train)
    predicted = fitted.predict(X_val)
    probability = fitted.predict_proba(X_val)[:, list(fitted.classes_).index(1)]
    fitted_holdout[name] = fitted
    metrics.append({
        "model": name,
        "f1": f1_score(y_val, predicted),
        "recall": recall_score(y_val, predicted),
        "roc_auc": roc_auc_score(y_val, probability),
    })
results = pd.DataFrame(metrics).set_index("model").sort_values("f1", ascending=False)
display(results)

In [ ]:
for name, fitted in fitted_holdout.items():
    print(f"\n{name}\n", classification_report(y_val, fitted.predict(X_val), digits=3))
    ConfusionMatrixDisplay.from_estimator(fitted, X_val, y_val, display_labels=[0, 1])
    plt.title(f"Validation confusion matrix: {name}")
    plt.show()

### Cross-validation on the training partition (optional)

Each fold fits a new imputer and classifier. The validation holdout remains unseen. Set `RUN_CV=True` for an extra estimate of F1 variability; cross-validation can be costly for the ensemble.

In [ ]:
RUN_CV = False
if RUN_CV:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for name, pipeline in models.items():
        scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1", n_jobs=1)
        print(f"{name}: F1 {scores.mean():.4f} ± {scores.std():.4f}; folds {scores}")

## 4. Feature inspection and selection

Mutual information is descriptive, and a low score alone does not establish ecological irrelevance.

In [ ]:
RUN_FEATURE_SELECTION = False
if RUN_FEATURE_SELECTION:
    training_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    X_train_imputed = training_imputer.fit_transform(X_train)
    scores = mutual_info_classif(X_train_imputed, y_train, random_state=RANDOM_STATE)
    display(pd.Series(scores, index=VARIABLES, name="mutual_information").sort_values(ascending=False))

    rfe_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("selection", RFE(RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1), n_features_to_select=10)),
        ("model", RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    rfe_pipeline.fit(X_train, y_train)
    selected_features = X_train.columns[rfe_pipeline.named_steps["selection"].support_].tolist()
    print("Selected features:", selected_features)
    print("Holdout F1:", f1_score(y_val, rfe_pipeline.predict(X_val)))

## 5. Hyperparameter search

In [ ]:
PARAM_GRIDS = {
    "random_forest": {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 10],
        "model__min_samples_leaf": [1, 3],
    },
    "lightgbm": {
        "model__n_estimators": [200, 400],
        "model__learning_rate": [0.03, 0.1],
        "model__num_leaves": [15, 31],
    },
    "xgboost": {
        "model__n_estimators": [200, 400],
        "model__learning_rate": [0.03, 0.1],
        "model__max_depth": [3, 5],
    },
}
RUN_TUNING = False
if RUN_TUNING:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    tuned_models = {}
    for name, grid in PARAM_GRIDS.items():
        search = GridSearchCV(models[name], grid, scoring="f1", cv=cv, n_jobs=1, refit=True)
        search.fit(X_train, y_train)
        tuned_models[name] = search.best_estimator_
        print(name, "CV F1:", round(search.best_score_, 4), "parameters:", search.best_params_)
        print("Holdout F1:", round(f1_score(y_val, search.predict(X_val)), 4))

## 6. Fit final model and predict test rows

Choose `SELECTED_MODEL` in the configuration cell using holdout results and the competition metric. If tuning was enabled, the selected tuned pipeline is refitted on every labeled row; otherwise a fresh baseline pipeline is fitted. `Target` must correspond to the competition's expected 0/1 label encoding. Run this section only after reviewing the validation comparison.

In [ ]:
if SELECTED_MODEL not in models:
    raise ValueError(f"Choose one of {list(models)}")
base_model = (tuned_models.get(SELECTED_MODEL, models[SELECTED_MODEL])
              if RUN_TUNING else models[SELECTED_MODEL])
final_model = clone(base_model).fit(X, y)
joblib.dump(final_model, OUTPUT_DIR / f"{SELECTED_MODEL}_pipeline.joblib")
test_predictions = final_model.predict(X_test)
if len(test_predictions) != len(test_locations):
    raise ValueError("Prediction count differs from test row count")
print("Prediction counts:", pd.Series(test_predictions).value_counts().sort_index().to_dict())

In [ ]:
submission = test_locations.drop(columns=["Latitude", "Longitude", "Target"], errors="ignore").copy()
submission["Target"] = test_predictions.astype(int)
submission_path = OUTPUT_DIR / f"submission_{SELECTED_MODEL}.csv"
submission.to_csv(submission_path, index=False)
print(f"Saved {len(submission)} rows to {submission_path}")
display(submission.head())